# Markov Chain Training (N=1 & N=2)
Ce notebook génère des modèles de Markov à partir du texte extrait (`bible_malagasy_brute.txt`).
Il crée deux fichiers JSON qui alimenteront l'auto-complétion (suggestions de mots) du Front-End.

In [2]:
import json
import re
from collections import defaultdict

# Fonction pour tokeniser et nettoyer le texte proprement
def clean_and_tokenize(text):
    # On remplace tout ce qui n'est pas une lettre, une apostrophe ou un espace par un espace
    text = re.sub(r'[^a-z\'\s\-]', ' ', text.lower())
    # On enlève les apostrophes qui ne sont pas entourées de lettres
    text = re.sub(r"(?<![a-z])\'|\'(?![a-z])", ' ', text)
    # Pareil pour les tirets (les mots rattachés comme lehibe-renirano, on les garde comme tels ou on les divise, 
    # en malgache on a beaucoup de traits d'union, on les enlève s'ils sont isolés)
    text = re.sub(r"(?<![a-z])\-|\-(?![a-z])", ' ', text)
    return text.split()

In [3]:
print("Chargement du texte d'entraînement...")
try:
    with open('bible_malagasy_brute.txt', 'r', encoding='utf-8') as f:
        text = f.read()
    tokens = clean_and_tokenize(text)
    print(f"{len(tokens)} tokens extraits.")
except FileNotFoundError:
    print("Erreur : Le fichier 'bible_malagasy_brute.txt' est introuvable.")
    tokens = []

Chargement du texte d'entraînement...
670180 tokens extraits.


In [4]:
if tokens:
    print("Construction du modèle N=1 (Ordre 1 : Bigrammes)...")
    markov_n1 = defaultdict(lambda: defaultdict(int))
    for i in range(len(tokens) - 1):
        w1, w2 = tokens[i], tokens[i+1]
        markov_n1[w1][w2] += 1

    print("Construction du modèle N=2 (Ordre 2 : Trigrammes)...")
    markov_n2 = defaultdict(lambda: defaultdict(int))
    for i in range(len(tokens) - 2):
        w1, w2, w3 = tokens[i], tokens[i+1], tokens[i+2]
        key = f"{w1} {w2}"
        markov_n2[key][w3] += 1
    print("Construction terminée.")

Construction du modèle N=1 (Ordre 1 : Bigrammes)...
Construction du modèle N=2 (Ordre 2 : Trigrammes)...
Construction terminée.


In [5]:
def filter_top_k(markov_dict, k=5):
    """
    Afin d'optimiser le poids des fichiers JSON pour le Front-End,
    nous ne conservons que les K mots les plus probables pour une clé donnée.
    """
    result = {}
    for key, next_words_dict in markov_dict.items():
        # Trie les mots suivants par leur occurrence (descendant)
        sorted_words = sorted(next_words_dict.items(), key=lambda x: x[1], reverse=True)
        # On garde uniquement les `k` premiers mots sous forme de liste de mots purs (sans les probabilités)
        result[key] = [word for word, count in sorted_words[:k]]
    return result

if tokens:
    print("Filtrage des 5 occurrences les plus probables...")
    n1_filtered = filter_top_k(markov_n1, k=5)
    n2_filtered = filter_top_k(markov_n2, k=5)
    
    print("Sauvegarde dans des fichiers JSON...")
    with open('markov_n1.json', 'w', encoding='utf-8') as f:
        # Mettre separators=(',', ':') pour réduire un peu la taille du fichier (minify)
        json.dump(n1_filtered, f, ensure_ascii=False, separators=(',', ':'))

    with open('markov_n2.json', 'w', encoding='utf-8') as f:
        json.dump(n2_filtered, f, ensure_ascii=False, separators=(',', ':'))
        
    print(f"✅ Fichiers générés avec succès !")
    print(f"-> markov_n1.json ({len(n1_filtered)} clés)")
    print(f"-> markov_n2.json ({len(n2_filtered)} clés)")

Filtrage des 5 occurrences les plus probables...
Sauvegarde dans des fichiers JSON...
✅ Fichiers générés avec succès !
-> markov_n1.json (27290 clés)
-> markov_n2.json (194670 clés)
